# 20. Hierarchical Indices

Two-tier retrieval: find the right section via summaries, then search chunks only inside it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Searching every small chunk of a large corpus is noisy: a chunk from the wrong section can look similar by accident. A **hierarchical index** searches in two tiers: first a small index of *section summaries* to find the right region, then the detailed chunks *inside only those sections*.

## How it works

1. Summarise each section with the LLM
2. Index the summaries (tier 1)
3. Index detailed chunks, each tagged with its section id (tier 2)
4. Query tier 1 to pick the top sections
5. Search tier 2 restricted to those sections

```text
 tier 1: [summary A] [summary B] [summary C] ...   <- search first (few vectors)
                         |
 tier 2: chunks of B only                          <- then search within (focused)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import numpy as np

sections = split_sections(load_text())
summaries = [llm(f"Summarise in one sentence:\n\n{body}") for _, body in sections]
summary_store = build_store([f"{h}: {s}" for (h, _), s in zip(sections, summaries)])

chunks, meta = [], []
for i, (heading, body) in enumerate(sections):
    for c in chunk_text(body, size=300, overlap=30):
        chunks.append(c); meta.append({"section": i})
chunk_store = build_store(chunks, meta)
print(len(sections), "summaries,", len(chunks), "chunks")

In [ ]:
def hierarchical_search(query, top_sections=2, k=3):
    wanted = {h["idx"] for h in summary_store.search(query, top_sections)}
    mask = np.array([m["section"] in wanted for m in meta])
    return chunk_store.search(query, k, mask=mask), wanted

query = "How does Microsoft GraphRAG answer global questions?"
hits, wanted = hierarchical_search(query)
print("Sections searched:", [sections[i][0] for i in wanted])
show(hits, width=100)
print(answer(query, [h["text"] for h in hits]))

Searching 2 small candidate sets is faster and less noisy than searching every chunk, especially on large corpora.

## Look at the summaries

In [ ]:
for (h, _), s in list(zip(sections, summaries))[:5]:
    print(f"{h}\n   {s}\n")

## Evaluate against flat search

In [ ]:
flat = chunk_store.search
hier = lambda q, k=3: hierarchical_search(q, top_sections=2, k=k)[0]
compare_retrievers({"flat chunk search": flat, "hierarchical (2 sections)": hier}, k=3)

## How many vectors are compared?
The hierarchy shrinks the search space - important at millions of chunks.

In [ ]:
per_section = {}
for m in meta: per_section[m["section"]] = per_section.get(m["section"], 0) + 1
flat_cost = len(chunks)
hier_cost = len(sections) + 2 * (sum(per_section.values()) / len(per_section))
print(f"flat: {flat_cost} comparisons   hierarchical: ~{hier_cost:.0f} comparisons")

## Failure mode: wrong section chosen
If tier 1 picks the wrong section the answer is unreachable. Raise `top_sections` to be safer.

In [ ]:
import pandas as pd
rows = []
for n in [1, 2, 3, 5]:
    r = evaluate_retriever(lambda q, k, n=n: hierarchical_search(q, top_sections=n, k=k)[0], k=3)
    rows.append({"top_sections": n, "hit@3": r["hit_rate@3"], "MRR": r["MRR"]})
pd.DataFrame(rows).set_index("top_sections")

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Smaller, cleaner search at each step | Summaries cost LLM calls |
| Scales to big corpora | Errors at tier 1 are unrecoverable |
| Summaries give a free 'table of contents' search | Needs structure (sections/documents) |

## When to use it

Large collections of long documents (books, reports, wikis) where the right document/section is the hard part.

## Experiments to try

- Add a third tier (document -> section -> chunk).
- Fall back to flat search when tier-1 scores are low.
- Use extractive summaries (first sentence) instead of LLM ones.

## Key takeaways

- Narrow first, then look closely.
- Tune how many sections to expand.
- Summaries double as navigation.

---
**Next:** [21. Dartboard Retrieval](../21_dartboard/dartboard.ipynb)